In [1]:
from sintactico_ast_ext import *
import semantico_ext_2_minicompilador

In [2]:
ts, errores = semantico_ext_2_minicompilador.simular_laberinto()


[Global] Declarado: x -> int
  self.ambitos = [{'x': 'int'}]

[Funcion test()] Entro ambito, declarado: a -> int

MOMENTO A - Justo despues de declarar int y = a * 2
  self.ambitos = [{'x': 'int'}, {'a': 'int', 'y': 'int'}]

[Bloque {}] Entro ambito, declarado: x -> float  (shadowing!)

  En 'y = y + x': tipo('y')=int, tipo('x')=float
  obtener_tipo_variable busca en reversed -> encuentra x:float primero
  Error 1: Tipos incompatibles en asignacion: 'y' es 'int' pero (y+x) produce 'float'

MOMENTO B - Justo antes de cerrar el bloque interno
  self.ambitos = [{'x': 'int'}, {'a': 'int', 'y': 'int'}, {'x': 'float'}]

[Bloque {}] Salio -> guardado en historial_ambitos

  En 'x = y + 1': Error 2: Reasignacion de 'x' sin tipo declarado: NodoAsignacion requiere tipo explicito

  En 'escribir(z)': Error 3: Error: Variable 'z' no identificada

MOMENTO C - Al llegar a escribir(z)
  self.ambitos = [{'x': 'int'}, {'a': 'int', 'y': 'int'}]

[Funcion test()] Salio -> guardado en historial_ambitos
 

In [3]:
print('Ámbitos activos finales:')
print(ts.ambitos)

Ámbitos activos finales:
[{'x': 'int'}]


In [4]:
print('Historial de ámbitos cerrados:')
print(ts.historial_ambitos)

Historial de ámbitos cerrados:
[{'x': 'float'}, {'a': 'int', 'y': 'int'}]


In [5]:
print('Funciones registradas:')
print(ts.funciones)

Funciones registradas:
{'test': ('void', [('a', 'int')])}


In [6]:
ts.imprimir_resumen_final(errores)

       RESUMEN FINAL DEL ANALISIS SEMANTICO

Ambito global actual (self.ambitos[0]):
   'x'             -> int

Historial de ambitos cerrados (2 registrados):
   Ambito historico [0]: {'x': 'float'}
   Ambito historico [1]: {'a': 'int', 'y': 'int'}

Funciones registradas:
   test() -> retorno: 'void', parametros: [('a', 'int')]

Errores semanticos detectados (3):
   Error 1: Tipos incompatibles en asignacion: 'y' es 'int' pero (y+x) produce 'float'
   Error 2: Reasignacion de 'x' sin tipo declarado: NodoAsignacion requiere tipo explicito
   Error 3: Error: Variable 'z' no identificada



In [7]:
# Construcción manual de un AST válido para probar el AnalizadorSemantico
# Equivale a: int suma(int a, int b) { int c = a + b; return c; }

nodo_suma = NodoFuncion(
    tipo_retorno=('KEYWORD', 'int'),
    nombre=('IDENTIFIER', 'suma'),
    parametros=[
        NodoParametro(('KEYWORD', 'int'), ('IDENTIFIER', 'a')),
        NodoParametro(('KEYWORD', 'int'), ('IDENTIFIER', 'b')),
    ],
    cuerpo=[
        NodoAsignacion(
            ('KEYWORD', 'int'),
            ('IDENTIFIER', 'c'),
            NodoOperacion(
                NodoIdentificador(('IDENTIFIER', 'a')),
                ('OPERATOR', '+'),
                NodoIdentificador(('IDENTIFIER', 'b'))
            )
        ),
        NodoRetorno(NodoIdentificador(('IDENTIFIER', 'c')))
    ]
)

nodo_main = NodoFuncion(
    tipo_retorno=('KEYWORD', 'int'),
    nombre=('IDENTIFIER', 'main'),
    parametros=[],
    cuerpo=[
        NodoAsignacion(
            ('KEYWORD', 'int'),
            ('IDENTIFIER', 'resultado'),
            NodoLlamadaFuncion('suma', [
                NodoNumero(('NUMBER', '3')),
                NodoNumero(('NUMBER', '5')),
            ])
        ),
        NodoRetorno(NodoNumero(('NUMBER', '0')))
    ]
)

arbol_ast = NodoPrograma(funciones=[nodo_suma], main=nodo_main)
print('AST construido correctamente')

AST construido correctamente


In [8]:
analizador = semantico_ext_2_minicompilador.AnalizadorSemantico()
analizador.analizar(arbol_ast)
print('Análisis semántico completado sin errores!')

Análisis semántico completado sin errores!


In [9]:
print('Tabla de símbolos — ámbitos:')
analizador.tabla_simbolos.ambitos

Tabla de símbolos — ámbitos:


[{}]

In [10]:
print('Funciones registradas:')
analizador.tabla_simbolos.funciones

Funciones registradas:


{'suma': ('int', [('a', 'int'), ('b', 'int')]), 'main': ('int', [])}

In [11]:
print('Historial de ámbitos:')
analizador.tabla_simbolos.historial_ambitos

Historial de ámbitos:


[{'a': 'int', 'b': 'int', 'c': 'int'}, {'resultado': 'int'}]